In [1]:
using LinearAlgebra, Polynomials, Plots
using Revise, DelimitedFiles, BenchmarkTools
using CloudAtlas, BifurcationKit, ChannelflowWrapper
using Dates
using Random
using Base.Threads

"""
    myreaddlm(filename, cc='%')

Read matrix or vector from a file, dropping comments marked with cc.
"""
function myreaddlm(filename; cc='%')
    X = readdlm(filename, comments=true, comment_char=cc)
    if size(X,2) == 1
        X = X[:,1]
    end
    X
end

macro suppress(ex)
    quote
        # Generate a unique name for the old stdout to avoid variable collision
        local old_stdout = stdout
        redirect_stdout(devnull)
        try
            # We use esc(ex) to run the expression in the caller's scope
            $(esc(ex))
        finally
            redirect_stdout(old_stdout)
        end
    end
end

sx, sy, sz, tx, tz = halfbox_symmetries()

pwd()

"/home/ebenq/Dev/julia/CloudAtlas.jl/notebooks/tw_fuzzing-tw2"

In [2]:
# Parameters
hookparams = SearchParams(ftol=1e-08, xtol=1e-12, Nnewton=30,Nhook=8,δ=0.01, verbosity=0)
Re = 300.0
cx0 = 0.000 # values from paper
cz0 = 0.009

α, γ = 2π/4.0, 2π/6.0                     # Fourier wavenumbers α, γ = 2π/Lx, 2π/Lz
H = [sz*tx]               # Generators of the symmetric subspace of TW2: sztx
normalize = true                    # Normalize the basis set or not?

# The DNS file to project from (ensure this path is correct)
dns_file = "TW1-2pi1piRe200-40x49x40.nc" 

# List of resolutions to test: [(J, K, L), ...]
# discretizations = [(1, 1, 1), (1, 1, 2), (1, 1, 3), (1, 2, 3), (1, 3, 5), (2, 4, 7), (3, 5, 9)]
discretizations = [(1, 1, 3), (1, 2, 3), (1, 3, 5), (2, 4, 7)]
# discretizations = [(1, 1, 3), (1, 2, 3)]

4-element Vector{Tuple{Int64, Int64, Int64}}:
 (1, 1, 3)
 (1, 2, 3)
 (1, 3, 5)
 (2, 4, 7)

In [3]:
"""
    fuzz_symmetry_space(discretizations, H, Re; attempts_per_level=10)

1. Loops through discretizations (J,K,L).
2. Generates random spectral guesses.
3. Attempts to converge using the low-dim ODE solver (`hookstepsolve`).
4. If the ODE solver converges, reconstructs the field and runs `findsoln`.
"""
function fuzz_symmetry_space(discretizations, H, Re; 
                             attempts_per_level=25, 
                             base_dir="fuzz_results",
                             noise_scale=1e-2,
                             symm_file = "./sztx.asc",
                             reference_path = "./TW1-2pi1piRe200-40x49x40.nc",
                             norm_threshold=1e-3,
                             xnorm = 0.40,
                             α=α,
                             γ=γ,
                             hookparams=hookparams,
                             T=20.0
    )
    
    # 1. Setup Directory
    mkpath(base_dir)
    
    # --- Thread Safety Tools ---
    io_lock = ReentrantLock()        # Prevents jumbled print output
    total_found = Atomic{Int}(0)     # Thread-safe counter

    reference_field_converted = "reference_field_$(α)_$(γ).nc"
    changegrid(reference_path, reference_field_converted; al=α, ga=γ)
    
    println("Starting Parallel Fuzz Search in $base_dir with $(nthreads()) threads")

    for (J, K, L) in discretizations
        # Use the lock to print cleanly
        lock(io_lock) do 
            println("\n" * "="^60)
            println("  Discretization: J=$J, K=$K, L=$L")
            println("="^60)
        end

        # Pre-calculate model for this level (shared by all threads)
        println("Our symmetry group is $H, α=$(α), and γ = $(γ)")
        model = TWModel(α, γ, J, K, L, H; normalize=false)
        m = length(model)

        @threads for i in 1:attempts_per_level
            
            # Generate Random Guess (Thread-local)
            x_guess = randn(m)
            x_guess = xnorm/norm(x_guess) * x_guess
            cx_guess = randn() * 0.1
            cz_guess = randn() * 0.1
            ξ_guess = [x_guess; cx_guess; cz_guess]

            # Try Low-Dimensional Solve
            # Note: We create new closures inside the loop so they are thread-safe
            f(ξ) = model.g(ξ, Re)
            Df(ξ) = model.Dg(ξ, Re)
            params = hookparams
            
            # Suppress output per thread to keep terminal clean
            ξ_star, converged = hookstepsolve(f, Df, ξ_guess, params)

            # Check convergence AND non-triviality
            solution_norm = norm(ξ_star[1:m])
            
            if converged && solution_norm > norm_threshold && ξ_star[end - 1] > 1e-7
                # Atomic add: safely increment counter
                atomic_add!(total_found, 1)
                
                # Lock output: safely print success message
                lock(io_lock) do
                    println("  [Thread $(threadid())] Hit! Converged at attempt #$i")
                end
                
                # 3. Promote to findsoln
                # Use 'i' in folder name to ensure unique paths
                timestamp = Dates.format(now(), "MM-DD-HHMMSS")
                sol_dir = joinpath(base_dir, "sol_$(J)_$(K)_$(L)_id$(i)_$(timestamp)")
                mkpath(sol_dir)
                
                guess_path = joinpath(sol_dir, "u_guess.nc")
                sigma_file = joinpath(sol_dir, "sigma.asc")
                
                # We lock file generation just to be safe with disk I/O bursts
                lock(io_lock) do
                    coeff2field(ξ_star[1:m], model.ijkl, reference_field_converted, guess_path)
                    save_sigma(model, ξ_star[end - 1], ξ_star[end], T, sigma_file)
                end
                
                try
                    # Run findsoln (External Binary)
                    # Note: Each thread launches its own process. 
                    findsoln(guess_path;
                        R = Re, eqb = true, xrel = model.keep_cx, zrel = model.keep_cz,
                        symms = abspath(symm_file), sigma = sigma_file, od = sol_dir,
                        T = T
                    )
                catch e
                    lock(io_lock) do
                        println("  [Thread $(threadid())] findsoln failed: $e")
                    end
                end
            end
        end
    end
end

function save_sigma(model::TWModel, cx::Real, cz::Real, T::Real, filename::String)
    open(filename, "w") do file
        az = cz ≈ 0 ? 0 : round(-cz * T, sigdigits=6)
        ax = cx ≈ 0 ? 0 : round(-cx * T, sigdigits=6)
        write(file, "% 1\n1 1 1 1 $(ax) $(az)")
    end
end

save_sigma (generic function with 1 method)

In [ ]:
# Parameters
hookparams = SearchParams(ftol=1e-08, xtol=1e-12, Nnewton=30,Nhook=8,δ=0.01, verbosity=0)
Re = 300.0
cx0 = 0.000 # values from paper
cz0 = 0.009

α, γ = 2π/4.0, 2π/6.0                     # Fourier wavenumbers α, γ = 2π/Lx, 2π/Lz
H = [sz*tx]               # Generators of the symmetric subspace of TW2: sztx
normalize = true                    # Normalize the basis set or not?

# The DNS file to project from (ensure this path is correct)
dns_file = "TW1-2pi1piRe200-40x49x40.nc" 

# List of resolutions to test: [(J, K, L), ...]
# discretizations = [(1, 1, 1), (1, 1, 2), (1, 1, 3), (1, 2, 3), (1, 3, 5), (2, 4, 7), (3, 5, 9)]
discretizations = [(1, 1, 3), (1, 2, 3), (1, 3, 5), (2, 4, 7)]
# discretizations = [(1, 1, 3), (1, 2, 3)]
fuzz_symmetry_space(discretizations, H, 300.0; attempts_per_level=25, α=α, γ=γ, hookparams=hookparams)

Leaving rescale
L2Norm(u0)  == 0.2792590266581449
L2Norm(u1)  == 0.2792590266581449
bcNorm(u0)  == 8.737886593015271e-17
bcNorm(u1)  == 1.001456466862179e-16
divNorm(u0) == 4.746636220033689e-16
divNorm(u1) == 6.392261886532839e-16
L2Norm(u2)  == 0.2792590266581449
divNorm(u2) == 6.271116005970482e-18
bcNorm(u2)  == 4.580081077120656e-17
Starting Parallel Fuzz Search in fuzz_results with 1 threads

  Discretization: J=1, K=1, L=3
Our symmetry group is Symmetry[Symmetry(1, 1, -1, 1//2, 0//1, 1)], α=1.5707963267948966, and γ = 1.0471975511965976
J,K,L,m == 1,1,3,33
(2J+1)(2K+1)(2L+1) + 1 == 64
Making matrices B, A1, A2, Cx, Cz...
Phase constraints: keep_cx = true, keep_cz = false
Making quadratic operator N...
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 
  [Thread 1] Hit! Converged at attempt #17
alpha, gamma == 1.570796326794897, 1.047197551196598
Nx, Ny, Nz == 40, 49, 40
Reading ijkl indices of basis set from file
reading N == 33 ijkl indic

f^T: ....10....20
Newton iteration number 0
Current state of Newton iteration:
   fcount_newton   == 1
   fcount_optimiza == 0
   L2Norm(x)       == 0.0251015
   L2Norm(dxN)     == 0
   L2Norm(dxOpt)   == 0
   L2Dist(x,x0)    == 0
gx == L2Norm(G(x)) : 
   initial  gx == 0.00136367
   previous gx == 0.00136367
   current  gx == 0.00136367
rx == 1/2 L2Norm2(G(x)) : 
   initial  rx == 9.29798e-07
   previous rx == 9.29798e-07
   current  rx == 9.29798e-07
         delta == 0.01
Newt,GMRES == 0,0, f^T: ....10....20 res == 0.382033
Newt,GMRES == 0,1, f^T: ....10....20 res == 0.218737
Newt,GMRES == 0,2, f^T: ....10....20 res == 0.042954
Newt,GMRES == 0,3, f^T: ....10....20 res == 0.0195232
Newt,GMRES == 0,4, f^T: ....10....20 res == 0.00700942
Newt,GMRES == 0,5, f^T: ....10....20 res == 0.00548156
Newt,GMRES == 0,6, f^T: ....10....20 res == 0.0041536
Newt,GMRES == 0,7, f^T: ....10....20 res == 0.00273451
Newt,GMRES == 0,8, f^T: ....10....20 res == 0.00251812
Newt,GMRES == 0,9, f^T: ....10...

f^T: ....10....20
Newton iteration number 0
Current state of Newton iteration:
   fcount_newton   == 1
   fcount_optimiza == 0
   L2Norm(x)       == 0.0195619
   L2Norm(dxN)     == 0
   L2Norm(dxOpt)   == 0
   L2Dist(x,x0)    == 0
gx == L2Norm(G(x)) : 
   initial  gx == 0.0010156
   previous gx == 0.0010156
   current  gx == 0.0010156
rx == 1/2 L2Norm2(G(x)) : 
   initial  rx == 5.15724e-07
   previous rx == 5.15724e-07
   current  rx == 5.15724e-07
         delta == 0.01
Newt,GMRES == 0,0, f^T: ....10....20 res == 0.363184
Newt,GMRES == 0,1, f^T: ....10....20 res == 0.24364
Newt,GMRES == 0,2, f^T: ....10....20 res == 0.0470234
Newt,GMRES == 0,3, f^T: ....10....20 res == 0.0136673
Newt,GMRES == 0,4, f^T: ....10....20 res == 0.00613493
Newt,GMRES == 0,5, f^T: ....10....20 res == 0.00440894
Newt,GMRES == 0,6, f^T: ....10....20 res == 0.0043986
Newt,GMRES == 0,7, f^T: ....10....20 res == 0.00427254
Newt,GMRES == 0,8, f^T: ....10....20 res == 0.00425116
Newt,GMRES == 0,9, f^T: ....10....20

f^T: ....10....20
Newton iteration number 0
Current state of Newton iteration:
   fcount_newton   == 1
   fcount_optimiza == 0
   L2Norm(x)       == 0.0895717
   L2Norm(dxN)     == 0
   L2Norm(dxOpt)   == 0
   L2Dist(x,x0)    == 0
gx == L2Norm(G(x)) : 
   initial  gx == 0.00774744
   previous gx == 0.00774744
   current  gx == 0.00774744
rx == 1/2 L2Norm2(G(x)) : 
   initial  rx == 3.00114e-05
   previous rx == 3.00114e-05
   current  rx == 3.00114e-05
         delta == 0.01
Newt,GMRES == 0,0, f^T: ....10....20 res == 0.18385
Newt,GMRES == 0,1, f^T: ....10....20 res == 0.113773
Newt,GMRES == 0,2, f^T: ....10....20 res == 0.0405121
Newt,GMRES == 0,3, f^T: ....10....20 res == 0.0154125
Newt,GMRES == 0,4, f^T: ....10....20 res == 0.0137934
Newt,GMRES == 0,5, f^T: ....10....20 res == 0.0135494
Newt,GMRES == 0,6, f^T: ....10....20 res == 0.013354
Newt,GMRES == 0,7, f^T: ....10....20 res == 0.0131094
Newt,GMRES == 0,8, f^T: ....10....20 res == 0.0130565
Newt,GMRES == 0,9, f^T: ....10....20 r

f^T: ....10....20
Newton iteration number 0
Current state of Newton iteration:
   fcount_newton   == 1
   fcount_optimiza == 0
   L2Norm(x)       == 0.0195619
   L2Norm(dxN)     == 0
   L2Norm(dxOpt)   == 0
   L2Dist(x,x0)    == 0
gx == L2Norm(G(x)) : 
   initial  gx == 0.00145493
   previous gx == 0.00145493
   current  gx == 0.00145493
rx == 1/2 L2Norm2(G(x)) : 
   initial  rx == 1.05841e-06
   previous rx == 1.05841e-06
   current  rx == 1.05841e-06
         delta == 0.01
Newt,GMRES == 0,0, f^T: ....10....20 res == 0.29858
Newt,GMRES == 0,1, f^T: ....10....20 res == 0.140891
Newt,GMRES == 0,2, f^T: ....10....20 res == 0.0220541
Newt,GMRES == 0,3, f^T: ....10....20 res == 0.00800455
Newt,GMRES == 0,4, f^T: ....10....20 res == 0.00271965
Newt,GMRES == 0,5, f^T: ....10....20 res == 0.00188406
Newt,GMRES == 0,6, f^T: ....10....20 res == 0.00181895
Newt,GMRES == 0,7, f^T: ....10....20 res == 0.00177114
Newt,GMRES == 0,8, f^T: ....10....20 res == 0.00176011
Newt,GMRES == 0,9, f^T: ....10.

f^T: ....10....20
Newton iteration number 0
Current state of Newton iteration:
   fcount_newton   == 1
   fcount_optimiza == 0
   L2Norm(x)       == 0.0195619
   L2Norm(dxN)     == 0
   L2Norm(dxOpt)   == 0
   L2Dist(x,x0)    == 0
gx == L2Norm(G(x)) : 
   initial  gx == 0.00124389
   previous gx == 0.00124389
   current  gx == 0.00124389
rx == 1/2 L2Norm2(G(x)) : 
   initial  rx == 7.73637e-07
   previous rx == 7.73637e-07
   current  rx == 7.73637e-07
         delta == 0.01
Newt,GMRES == 0,0, f^T: ....10....20 res == 0.336168
Newt,GMRES == 0,1, f^T: ....10....20 res == 0.220082
Newt,GMRES == 0,2, f^T: ....10....20 res == 0.0265741
Newt,GMRES == 0,3, f^T: ....10....20 res == 0.0126739
Newt,GMRES == 0,4, f^T: ....10....20 res == 0.00369061
Newt,GMRES == 0,5, f^T: ....10....20 res == 0.00335398
Newt,GMRES == 0,6, f^T: ....10....20 res == 0.00333996
Newt,GMRES == 0,7, f^T: ....10....20 res == 0.00329588
Newt,GMRES == 0,8, f^T: ....10....20 res == 0.00327591
Newt,GMRES == 0,9, f^T: ....10.

f^T: ....10....20
Newton iteration number 0
Current state of Newton iteration:
   fcount_newton   == 1
   fcount_optimiza == 0
   L2Norm(x)       == 0.0195619
   L2Norm(dxN)     == 0
   L2Norm(dxOpt)   == 0
   L2Dist(x,x0)    == 0
gx == L2Norm(G(x)) : 
   initial  gx == 0.00131167
   previous gx == 0.00131167
   current  gx == 0.00131167
rx == 1/2 L2Norm2(G(x)) : 
   initial  rx == 8.60241e-07
   previous rx == 8.60241e-07
   current  rx == 8.60241e-07
         delta == 0.01
Newt,GMRES == 0,0, f^T: ....10....20 res == 0.32556
Newt,GMRES == 0,1, f^T: ....10....20 res == 0.204726
Newt,GMRES == 0,2, f^T: ....10....20 res == 0.0243303
Newt,GMRES == 0,3, f^T: ....10....20 res == 0.0121726
Newt,GMRES == 0,4, f^T: ....10....20 res == 0.00337812
Newt,GMRES == 0,5, f^T: ....10....20 res == 0.00296598
Newt,GMRES == 0,6, f^T: ....10....20 res == 0.00292765
Newt,GMRES == 0,7, f^T: ....10....20 res == 0.0028828
Newt,GMRES == 0,8, f^T: ....10....20 res == 0.00287834
Newt,GMRES == 0,9, f^T: ....10...

f^T: ...